# <span style="color:red">**Question Answering**</span>

In this project, you will learn how to finetune and use pretrained **deep learning models**, a prominent technique used nowadays in the Natural Language Processing field. These models of interest are usually called **Transfomers**.

In particular, you will learn how to develop a model of the **BERT** (Bidirectionnal Encoder Representation Transformer) family to perform **Question Answering**.

**Question Answering** is a specific field of Computational Linguistics that can be described as the ability of Computer to answer a question based on a given context.
Although there are different ways to model this task, this project considers a relatively simple setup with three main elements:
- The **Question**: something you ask and expect an answer for. All the questions in this project end with a question mark ("?").
- The **Context**: a text with one or more sentences in which the answer to the question is expected to be located.
- The **Answer**: the part of the context considered as the answer to the question and the position of the starting word of the answer in the context. Thus, it is composed of 2 elements.


Here is an example of a question, the context and the answer:

* **Question**: What is the oldest structure at Notre Dame?
* **Context**: The university is the major seat of the Congregation of Holy Cross (albeit not its official headquarters, which are in Rome). Its main seminary, Moreau Seminary, is located on the campus across St. Joseph lake from the Main Building. Old College, the oldest building on campus and located near the shore of St. Mary lake, houses undergraduate seminarians. Retired priests and brothers reside in Fatima House (a former retreat center), Holy Cross House, as well as Columba Hall near the Grotto. The university through the Moreau Seminary has ties to theologian Frederick Buechner. While not Catholic, Buechner has praised writers from Notre Dame and Moreau Seminary created a Buechner Prize for Preaching.
* **Answer**:
    * **Text**: Old College
    * **Starting word position**: 234


**NOTE**: you must not (and can not) edit this notebook as it is the template shared with everyone. You can download a copy and edit it by doing File (upper left corner) then Save a copy in Drive. You can also just Download it as a .ipynb if you prefer.

## <span style="color:#ff5500">Software requirements</span >

The deep learning library used in this project is PyTorch, a free and open-source framework.

This project can either be done on your computer or on Google Colab, but for the models to be trained in reasonable amount of time, it **requires** a GPU. Such a GPU will only be necessary for the **second part** of this project.

###  <span style="color:#ff8800">A. Recommended choice: using Colab</span >

Using Google Colab to complete the assignment is your best bet for a hassle
free experience on this project.

In order to access data stored in your drive, in Colab, you need to mount your google drive to be able to have a repository where you can store data. Here is the snippet to do it properly on Colab.

Note: you will need to grant it all the permissions. You will also receive a warning on your email about a connection from another device. This is normal and expected.

In [ ]:
# Execute this to be able to access the drive
from google.colab import drive
drive.mount('/content/drive')

### <span style="color:#ff8800">B. Other choice: Locally on your machine</span>

Should you decide to install the software on your own machine; then the
**very first step** of this project would consist in installing [PyTorch][2]
on your machine. You can use the generated command (while clicking on PyTorch link) but first read the next lines.

Please note that if you don't have a GPU with 16GB of RAM, you should not try to train models you will built in this projet locally. Also note that depending on the type of GPU (NVIDIA vs AMD), your installation of PyTorch will differ a bit so be cautious about the compute platform you have. Finally, note that the *torchaudio* and *torchvision* libraries are not required for this project.

### <span style="color:#ff8800">C. Libraries</span>

Several other libraries are required for this part of project:
* [numpy][3]
* [transformers][4]
* [datasets][5]


##### <span style="color:#ffcc00">Installation of libraries in Colab </span>

To install all those libraries on Colab direclty, you can use this snippet in a python cell of your notebook:



[2]:<https://pytorch.org/get-started/locally/>
[3]:<https://numpy.org/>
[4]:<https://huggingface.co/docs/transformers/index>
[5]:<https://huggingface.co/docs/datasets/index>


In [ ]:
# Execute this to be able to install the required libraries for this part of the project
!pip install torch numpy datasets transformers

## <span style="color:#ff5500">**Section one:** Data acquisition

In the first part of the project, you will learn how to load and to preprocess a dataset with the Huggingface libraries. A Dataset from Huggingface contains the data in a data structure similar to a Python DataFrame.

### <span style="color:#ff8800">Data obtention</span>

To get access to an open-source dataset, an easy method consists in using the Huggingface library called **datasets**. For this project, we will use the [Stanford Question Answering Dataset (SQuAD)][1], a question answering dataset used to benchmark question answering models.

Execute the code in the next cell to load the dataset.

[1]:<https://huggingface.co/datasets/squad>

In [ ]:
from datasets import load_dataset
squad_dataset = load_dataset("squad")

## Very important
SQuAD dataset contains some annotation errors.
To solve this issue, consider the filter below to remove those elements from both train and validation splits. You will need to download this [csv file][1] in order to be able to load the indices to remove (this file is also available from the [first task][2] of the project on Inginious).
Upload it to your Google Drive to have access to it via Colab.

[1]:<https://inginious.info.ucl.ac.be/course/LINFO2263/project3a/indices.csv>
[2]:<https://inginious.info.ucl.ac.be/course/LINFO2263/project3a>

In [ ]:
import csv
path = "/content/drive/MyDrive/indices.csv" 
indices_to_remove_train = []
indices_to_remove_valid = []
with open(path) as f:
    reader_obj = csv.reader(f)
    indices_to_remove_train = list(map(int, next(reader_obj)))
    indices_to_remove_valid = list(map(int, next(reader_obj)))

train_dataset = squad_dataset["train"]
train_dataset = train_dataset.filter(lambda example, idx: idx not in indices_to_remove_train, with_indices=True)

validation_dataset = squad_dataset["validation"]
validation_dataset = validation_dataset.filter(lambda example, idx: idx not in indices_to_remove_valid, with_indices=True)

### <span style="color:#ff8800"> Question 1: Dataset size</span>
Here, the squad_dataset is splitted into two splits: 1) the Training split, to train the model, and 2) the Validation split, to evaluate its performance across epochs of training.

Both splits contains data that are stored as rows in the dataset. Each example (= each row) contains several fields (id, title, context, question, answers) that we refer to as the **features** of the dataset.

>**QUESTION**
>
>What are the number of examples of these two splits in the filtered dataset?

Your answer must be formatted as a Python dictionary with 2 keys, "train" and "validation", with their respective length in the filtered dataset. Report your answer on Inginious.


In [ ]:
{"train":2000, "validation":2000}

### <span style="color:#ff8800">Question 2: Content</span>

>**QUESTION**
>
>What is the value of the feature "id" of the example at index 268 (hence the 269th element) of the filtered train dataset? What is the value of the feature "title" of this example?

Your answer must be formatted as a Python dictionary with 2 keys, "id" and "title", with their respective value. Report your answer on Inginious.


In [ ]:
{"id":'6523ab89', 'title':'UCLouvain'}

## <span style="color:#ff5500">**Section two**: Data tokenization</span>

### <span style="color:#ff8800">**Introduction to tokenizers**</span>

Before getting into what you will do, let's explain what a
[tokenizer from transformers library][8] does. It allows you to:

1) Split word strings into (sub-)word tokens and back

2) Convert (sub-)word tokens into ids (= integers) and back

3) Add new tokens to the vocabulary (ex: [CLS], [SEP], [PAD], ...)

The combination of steps 1 and 2 is called encoding (from words to ids) or decoding (from ids to words).

[8]:<https://huggingface.co/docs/transformers/main_classes/tokenizer>

### <span style="color:#ff8800">**Task specific tokenizer**</span>

Each pretrained transformer model has its corresponding tokenizer. This allows the model and the tokenizer to consider the same vocabulary and other preprocessing parameters.

As mentionned in the introduction of this project, you will use a BERT model. More precisely, you will use the DistilBERT model (a [distilled][1], smaller version of the BERT model).

[1]:<https://en.wikipedia.org/wiki/Knowledge_distillation>

You can then load the tokenizer corresponding to this model with the code cell below.

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


### <span style="color:#ff8800">**Tokenizer parameters**</span>

The [tokenizer][1] from HuggingFace can be called to encode both the question and context of an example (= one specific row of the dataset) using the following template:

```python
tokenized_example = tokenizer(example["question"], example["context"], ...)

```
Moreover, in order to work correctly, the tokenizer must receive different parameters.
For the sake of reproducibility and to realize this project, we ask you to fix some of these parameters for the next questions. You can click on their names below to access their description in the documentation.

- [stride][2]: 32
- [max_length][3]: 144
- [padding][4]: "max_length"
- [return_overflowing_tokens][5]: True

The questions 1 and 2 below will make you browse through the documentation to determine the two remaining parameters to set when using the tokenizer.

Once you have found all the parameters, you will be able to use the tokenizer to encode the whole dataset into *ids* that the DistilBERT model can understand.
If you set the correct parameters, the tokenization obtained for each example will look like this:

**<p align=center>[CLS] question tokens [SEP] context tokens [SEP]</p>**






[1]:<https://huggingface.co/docs/transformers/model_doc/distilbert#transformers.DistilBertTokenizer>
[2]:<https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.__call__.stride>
[3]:<https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.__call__.max_length>
[4]:<https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.__call__.padding>
[5]:<https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.__call__.padding>



### <span style="color:#ff8800">Question 1: Truncation</span>

In the dataset, it is possible that some examples have a context that is longer than the specified *max_length*. It is therefore necessary that the tokenizer truncates (or cuts) these contexts into several tokenized outputs.

Here, we consider that only the context can exceed this *max_length*, not the question. Thus, you **should not** truncate the question but only the context whenever it exceeds *max_length* .

> **QUESTION**
>
> Give the value of the *truncation* parameter such that you truncate only the context (when needed) and not the question.

Your answer must be formatted as a Python dictionary with the key "truncation" and its corresponding value. Report your answer on Inginious.


In [ ]:
{"truncation": "value_of_parameter"}

###  <span style="color:#ff8800">Question 2: Mapping</span>

As mentioned in the introduction, the dataset contains, for each example, the start index (i.e. the index of the first character) of the answer to the question in the original context.
However, after tokenization, the length of each context may have increased since some words may be have been fragmented into several sub-word tokens (check [the tokenizer of the version of DistilBERT we use][2], a modified version of [WordPiece][3]). Therefore, a tokenized context may include a larger number of tokens than the number of words in the original context.

Because of this fragmentation, finding the sub-word token that is the start of the answer is a bit more  difficult.
Thus, it is necessary to have a **mapping** that links the sub-word tokens obtained with the tokenizer to their original start and end indices.

**Example**: Consider the sentence: "This subword is a token" and consider its tokenized version: ['This', 'sub', '##word', 'is', 'a', 'token']. The mapping of the *start* and *end* **character indices** would be: [(0, 4), (5, 8), (8, 12), (13, 15), (16, 17), (18, 23)] where each element is a range, that is the second value is not included. The mapping of the second token, "sub", is (5, 8) because it starts at index 5 and ends at index 8 (not included) in the original sentence. The third token, "##word", has been modified by the tokenizer but the mapping allows us to retrieve that it starts at index 8 and ends at index 12 (not included) in the original sentence. Note that the length of this token is actually longer because of the extra characters "##" used to denote a *wordpiece*.


Hopefully, a parameter of the tokenizer allows to do this mapping during the tokenization process.

> **QUESTION**
>
> What is the name of the parameter of the tokenizer that allows to retreive such mapping?

Give the name of the parameter in a string. Report your answer on Inginious.


[2]:<https://huggingface.co/docs/transformers/model_doc/distilbert#transformers.DistilBertTokenizer>
[3]:<https://paperswithcode.com/method/wordpiece>

In [ ]:
"name_of_parameter"

### <span style="color:#ff8800">Question 3: Keys of tokenized dataset</span>

You have now all the parameters needed to call the tokenizer and continue this project as intended!

To make sure everything is working well, use the tokenizer on a single example of the dataset with the parameters defined above.

The output of the tokenizer should be a Python dictionary.

> **QUESTION**
>
> What are the keys of the dictionary returned by the tokenizer?

Your answer must be a Python set containing the keys of the output dictionary. Report your answer on Inginious.


**Note:** The *tokenizer* variable created above is a callable object that requires several parameters. Therefore, you can directly call tokenizer(p1, p2, p3, ...) with all the parameters you need to set.


In [ ]:
example = ...
output = tokenizer(example["question"], example["context"], ...)


In [ ]:
{"name_of_a_key", "name_of_another_key", "name_of_a_third_key", ...}

### <span style="color:#ff8800">Tokenize function</span>

Instead of applying it sequentially to each example, you can call the tokenizer on the whole dataset directly!

In the incomplete function below, you are asked to complete the missing parts with what you have learned in the previous questions in order to tokenize the entire dataset given in the arguments.

The arguments of the function are the following:
- dataset: the data to tokenize. In this case, this represents either the train or the validation split.
- tokenizer: the tokenizer object you use to process the dataset (recall that you have loaded the tokenizer coming from a pretrained model)
- stride: the number of overlapping tokens between two successive chunks, default=32
- max_length: the maximum length to be treated by the tokenizer, default=144

The 'map' function applies the 'encode' function to each example in the dataset. As a result, it outputs a new version of the same dataset with some additional columns, which are precisely the outputs of the tokenizer (see question 3).



In [ ]:
def tokenize(dataset, tokenizer, stride=32, max_length=144):
    def encode(example):
        return tokenizer( example["question"],
                          example["context"],
                          return_offsets_mapping=True,
                         truncation="only_second",
                         stride=stride,
                         max_length=max_length,
                         padding="max_length",
                         return_overflowing_tokens=True,
                         )

    # Map function encode to each question+context of the dataset one by one
    tokenized_dataset = dataset.map(encode)

    # Return tokenized dataset
    return tokenized_dataset


### <span style="color:#ff8800">Question 4: Example</span>

It is now time to use (and test!) your *tokenize* function!
To ensure that your function executes correctly, apply it on the first 6 examples of the filtered train split. We refer to this reduced train split as the **toy dataset**.

In this question, we ask you to analyze the **Dataset** obtained as output by using the *tokenize* function on the toy dataset. You need to understand the meaning of each feature of this output to retreive the relevant information! Do not hesitate to look at the [documentation][1]. Note also that, due to the *map* function, this Dataset contains both the features from the input (id, question, ...) and from the tokenizer (that you found in the previous question).

[1]:<https://huggingface.co/docs/transformers/model_doc/distilbert#transformers.DistilBertTokenizer>

More specifically, consider the following points:
- Due to the truncation and the maximum length that can be handled by the tokenizer and the model, some examples may see their context cut into different segments.
- The tokenizer encodes the text into ids
- The tokenizer encodes both the question and the context together

> **QUESTION**
>
> 1) In how many **segments** has the row at index 3 (hence the 4th example, i.e. a question+context pair) of the toy dataset been cut into due to the truncation? Your answer must be a single int.
>
> 2) What are the 6 first **token ids** (including the id for [CLS]) of the encoded **question** of this row at index 3? Your answer must be a list of ints of 6 elements, each one corresponding to an id.
>
> 3) Which **(sub-)word** tokens (=character strings) do these 6 first ids correspond to? Your answer must be a list of string with 6 elements, each one corresponding to a subword.
>
> 4) What are the 6 first **token ids** (excluding the id for [SEP]) of the truncated **context** in the second segment of this row at index 3? Your answer must be a list of ints of 6 elements, each one corresponding to an id.
>
> 5) Which **(sub-)word** tokens do these 6 first ids (= 6 character strings appearing after [SEP]) correspond to? Your answer must be a list of string with 6 elements, each one corresponding to a subword.



Format your question using a Python dictionary with the following keys: "n_parts", "ids_question", "subwords_question", "ids_context", "subwords_context". Indicate as values your answers for each respective question. Report your answer on Inginious.

Hint: a function of the [pretrained tokenizer][2] can convert *ids* to sub-word tokens. Which one is it?

[2]:<https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer>





In [ ]:
toy_dataset = load_dataset("squad", split='train[:6]')

In [ ]:
tokenized_toy_dataset = tokenize(toy_dataset, tokenizer)
example_4 = tokenized_toy_dataset.select([3])  
print(example_4)
n_parts = len(example_4["input_ids"][0]) 
ids_question = example_4["input_ids"][0][0][:6]  
subwords_question = tokenizer.convert_ids_to_tokens(ids_question)
sep_index = example_4["input_ids"][0][1].index(tokenizer.sep_token_id)  
ids_context = example_4["input_ids"][0][1][sep_index + 1 : sep_index + 7]
subwords_context = tokenizer.convert_ids_to_tokens(ids_context)

output = {
    "n_parts": n_parts,
    "ids_question": ids_question,
    "subwords_question": subwords_question,
    "ids_context": ids_context,
    "subwords_context": subwords_context,
}

print(output)

### <span style="color:#ff8800">Question 5: Tokenized dataset size</span>

In the previous questions, you have worked with examples to understand all the parameters used in the tokenizer. Now, you should be able to tokenize the whole dataset!

To do this and for the **part 2** of this project, we consider the following values for the *stride* and *max_length* parameters:
- stride = 150
- max_length = 512

Once you have tokenized a dataset, the number of segments (recall that one segment is a question + a possibly *truncated* context) could be higher than the number of examples in the dataset, precisely due to the truncation process.

> **QUESTION**
>
>What is the difference of size of each split? Basically, you must compute the difference
*total_number_of_segments_split - total_number_of_examples* for each split, train and validation.


Format your answer using a Python dictionary with two keys, "train" and "validation", and their corresponding value for the size difference. Report your answer on Inginious.




In [ ]:
tokenized_train_dataset = tokenize(train_dataset, tokenizer ,stride=150,max_length=512)
tokenized_validation_dataset = tokenize(validation_dataset, tokenizer, stride=150,max_length=512)

train_num_examples = len(squad_dataset["train"])
train_num_segments = len(tokenized_train)

validation_num_examples = len(squad_dataset["validation"])
validation_num_segments = len(tokenized_validation)

train_difference = train_num_segments - train_num_examples
validation_difference = validation_num_segments - validation_num_examples

# Output results
output = {
    "train_difference": train_difference,
    "validation_difference": validation_difference,
}

print(output)

In [ ]:
print(f"Number of original examples: {len(train_dataset)}")
print(f"Number of tokenized segments: {tokenized_train.num_rows}")
print(tokenized_train[0])  # Inspect a tokenized example
max_context_length_train = max(len(example["context"]) for example in train_dataset)
max_context_length_valid = max(len(example["context"]) for example in validation_dataset)

print(f"Max context length (train): {max_context_length_train}")
print(f"Max context length (validation): {max_context_length_valid}")

{'train': ..., 'validation': ...}

You can now save the tokenized versions of the dataset to use them in the 2nd part of this project!

In [ ]:
# Execute this code
tokenized_train_dataset.save_to_disk("path_to_location_in_the_drive_or_locally")      # TODO: replace with the path where you want to save the tokenized version of the train split
tokenized_validation_dataset.save_to_disk("path_to_location_in_the_drive_or_locally") # TODO: replace with the path where you want to save the tokenized version of the validation split